# Scaling Laws and Compute-Optimal Training

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 13.01, 05.03, 12.04 |
| Optional | 13.08 (double descent — for theoretical connection) |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/13_Statistical_Learning_Theory/09_scaling_laws.ipynb)

---

## 🎯 Learning Objective

Understand the mathematical structure of neural scaling laws — power-law relationships between
loss and model size, dataset size, and compute — and learn how to derive compute-optimal
training allocations (the Chinchilla result). Examine the debate around emergent abilities as
possible metric artifacts, and connect scaling laws to the eigenspectrum theory from `08.04`.

---

## 📐 Theory

### Empirical scaling laws (Kaplan et al., 2020)

Large-scale experiments on autoregressive language models reveal that test loss $L$ follows
**power-law** relationships with three independent axes of scale, each holding the other two
fixed and sufficiently large:

$$L(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}, \qquad
  L(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D}, \qquad
  L(C) \approx \left(\frac{C_c}{C}\right)^{\alpha_C}$$

where:
- $N$ = number of model parameters (excluding embeddings),
- $D$ = number of training tokens (dataset size),
- $C$ = compute budget in FLOPs (floating-point operations),
- $N_c, D_c, C_c$ = characteristic scales at which loss equals 1 nat,
- $\alpha_N \approx 0.076$, $\alpha_D \approx 0.095$, $\alpha_C \approx 0.050$ are the fitted
  power-law exponents.

On a **log-log plot** these become straight lines: $\log L = -\alpha \log X + \text{const}$.
The key empirical finding is that the exponents are *small* — diminishing returns set in
slowly, meaning each $10\times$ increase in scale still yields a meaningful loss reduction.

When all three resources are varied simultaneously, Kaplan et al. propose a **combined law**:

$$L(N, D) = \left[\left(\frac{N_c}{N}\right)^{\alpha_N / \alpha_D} + \frac{D_c}{D}\right]^{\alpha_D}$$

This has the correct limiting behavior: as $D \to \infty$ the second term vanishes and we
recover $L(N)$; as $N \to \infty$ the first term vanishes and we recover $L(D)$.

### Chinchilla scaling (Hoffmann et al., 2022)

Given a fixed compute budget $C$, the fundamental question is how to allocate it between model
size $N$ and training tokens $D$. The compute constraint from `12.04` is approximately:

$$C \approx 6ND$$

(each token requires $\approx 6N$ FLOPs for a forward + backward pass in a dense transformer).
Hoffmann et al. fit three independent estimation approaches and find that the compute-optimal
allocation satisfies:

$$N_{\text{opt}} \propto C^a, \qquad D_{\text{opt}} \propto C^b, \qquad a + b = 1, \quad a \approx b \approx 0.5$$

This means **parameters and data should scale equally**: doubling compute should double both $N$
and $D$, not put all resources into a bigger model. Formally, from the loss decomposition
$L(N,D) = A N^{-\alpha} + B D^{-\beta} + L_\infty$ (with irreducible loss $L_\infty$), we form
the Lagrangian (recall `04.05`):

$$\mathcal{L}(N, D, \lambda) = A N^{-\alpha} + B D^{-\beta} + L_\infty + \lambda(6ND - C)$$

Setting $\frac{\partial \mathcal{L}}{\partial N} = 0$ and $\frac{\partial \mathcal{L}}{\partial D} = 0$:

$$-A\alpha N^{-\alpha - 1} + 6\lambda D = 0 \qquad \Longrightarrow \qquad \lambda = \frac{A\alpha N^{-\alpha - 1}}{6D}$$

$$-B\beta D^{-\beta - 1} + 6\lambda N = 0 \qquad \Longrightarrow \qquad \lambda = \frac{B\beta D^{-\beta - 1}}{6N}$$

Equating the two expressions for $\lambda$:

$$\frac{A\alpha}{N^{\alpha + 1} D} = \frac{B\beta}{D^{\beta + 1} N}
\qquad \Longrightarrow \qquad
\frac{A\alpha}{B\beta} = \frac{N^\alpha}{D^\beta}$$

Substituting $D = C / (6N)$ and solving yields the Chinchilla result:

$$N_{\text{opt}} = G \cdot C^{a}, \quad D_{\text{opt}} = \frac{C}{6 N_{\text{opt}}},
\quad a = \frac{\beta}{\alpha + \beta}$$

where $G$ is a constant depending on $A, B, \alpha, \beta$. With Hoffmann et al.'s fitted
$\alpha \approx \beta$, we get $a \approx 0.5$.

**Key insight vs Kaplan.** Kaplan et al. recommended scaling $N$ faster than $D$ (roughly
$N \propto C^{0.73}$, $D \propto C^{0.27}$). The Chinchilla result corrects this: Kaplan's
fits were confounded by not training to convergence at each scale point, biasing the $D$
exponent downward. The practical consequence was enormous: by the Chinchilla analysis of
Hoffmann et al. (2022, cited in Further Reading), GPT-3 (175B params, 300B tokens)
was substantially *undertrained* by Chinchilla standards; an equally good model could have
been trained with $\approx 70$B params on $\approx 1.4$T tokens at the same compute cost —
the parameters and tokens of Chinchilla itself, Hoffmann et al. (2022).

### Why power laws? Connections to spectral theory

Power-law scaling is not a coincidence — it arises from the **eigenvalue spectrum** of the data
covariance matrix (recall `08.04`). If the eigenvalues of the population covariance
$\Sigma$ decay as $\lambda_k \sim k^{-\gamma}$ for some spectral exponent $\gamma > 1$, then a
model with $N$ parameters can effectively capture the top $\sim N$ eigendirections, incurring
residual loss proportional to the tail sum:

$$L(N) \propto \sum_{k > N} \lambda_k \approx \int_N^\infty x^{-\gamma} dx = \frac{N^{1-\gamma}}{\gamma - 1} \propto N^{-(\gamma - 1)}$$

This gives $\alpha_N = \gamma - 1$. Similarly, $D$ tokens provide $\sim D$ effective samples
for estimating each eigendirection, and the estimation error per direction contributes a
$D^{-\beta}$ term. The power-law structure is thus a direct consequence of the *heavy-tailed*
nature of natural data distributions — a connection to random matrix theory (`08.04`) and the
universality of spectral decay in natural signals.

### Emergent abilities and phase transitions

Some capabilities appear to "emerge" abruptly at specific model scales — near-zero performance
below a threshold, then rapid improvement. **Schaeffer et al. (2023)** argue that many such
transitions are **metric artifacts** rather than true phase transitions. The core argument:

Suppose the underlying capability $p(N)$ is a smooth sigmoid:

$$p(N) = \sigma(k(\log N - \log N_0)) = \frac{1}{1 + e^{-k(\log N - \log N_0)}}$$

where $k$ controls steepness and $N_0$ is the inflection point. If we measure performance
with **accuracy** (a nonlinear, thresholded metric), we observe:

$$\text{Accuracy}(N) = \begin{cases} \approx 0 & \text{if } p(N) < 1/K \text{ (random guessing among } K \text{ choices)} \\ \text{rapidly increasing} & \text{otherwise} \end{cases}$$

The same smooth $p(N)$ measured by **log-loss** (a smooth metric) shows no discontinuity:

$$\text{Log-loss}(N) = -\log p(N)$$

decreases continuously. The apparent "emergence" is thus the *composition* of a smooth
function with a nonlinear metric — the discontinuity is in the *metric*, not the *capability*.

### Beyond language models: universality of scaling laws

Power-law scaling has been observed across modalities:
- **Vision** (ViT): power-law error scaling with model/data/compute, with small
  reported exponents (Zhai et al., 2022, *Scaling Vision Transformers*; cited in Further Reading)
- **Speech** (Whisper-family): similar power-law behavior in WER vs model size
- **Biology** (protein structure prediction): scaling in accuracy vs training set size
- **Multi-modal** (CLIP, Flamingo): joint scaling in both modalities

This cross-modal universality was documented by Henighan et al. (2020, cited in Further Reading).
It *suggests* the phenomenon is driven by the statistical structure of natural data — the
heavy-tailed eigenspectrum connection above is one proposed explanation (Bahri et al., 2021),
not a settled result.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

We reproduce the key empirical observation from Kaplan et al. on synthetic data: log-log
plots of loss vs model size ($N$), dataset size ($D$), and compute ($C$) all produce straight
lines — the signature of power-law scaling. The second panel overlays the Chinchilla
compute-optimal frontier against the Kaplan recommendation, showing how the two prescriptions
diverge as the compute budget grows. The third panel demonstrates the Schaeffer et al.
argument: the same smooth underlying capability produces an apparent "phase transition" under
accuracy but a smooth curve under log-loss.

In [ ]:
BLUE, RED, GREEN, PURPLE = "#4C72B0", "#C44E52", "#55A868", "#8172B2"

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# --- Panel 1: Power-law scaling (Kaplan-style) ---
ax = axes[0]
alpha_N, alpha_D, alpha_C = 0.076, 0.095, 0.050
Nc, Dc, Cc = 8.8e13, 5.4e13, 3.1e8

N_vals = np.logspace(6, 11, 50)
D_vals = np.logspace(7, 12, 50)
C_vals = np.logspace(17, 24, 50)

L_N = (Nc / N_vals) ** alpha_N
L_D = (Dc / D_vals) ** alpha_D
L_C = (Cc / C_vals) ** alpha_C

ax.loglog(N_vals, L_N, color=BLUE, lw=2.5, label=r'$L(N) = (N_c/N)^{0.076}$')
ax.loglog(D_vals, L_D, color=RED, lw=2.5, ls='--', label=r'$L(D) = (D_c/D)^{0.095}$')
ax.loglog(C_vals, L_C, color=GREEN, lw=2.5, ls=':', label=r'$L(C) = (C_c/C)^{0.050}$')
ax.set_xlabel('Scale (N, D, or C)'); ax.set_ylabel('Loss (nats)')
ax.set_title('Power-Law Scaling (Kaplan et al.)')
ax.legend(fontsize=8, loc='upper right')

# --- Panel 2: Chinchilla vs Kaplan compute-optimal frontier ---
ax = axes[1]
C_budget = np.logspace(18, 25, 100)

# Chinchilla: N_opt ~ C^0.50, D_opt = C / (6 * N_opt)
G_chin = 0.2  # proportionality constant (approximate)
N_chin = G_chin * C_budget ** 0.50
D_chin = C_budget / (6 * N_chin)

# Kaplan: N_opt ~ C^0.73, D_opt = C / (6 * N_opt)
G_kap = 0.0003
N_kap = G_kap * C_budget ** 0.73
D_kap = C_budget / (6 * N_kap)

ax.loglog(C_budget, N_chin, color=BLUE, lw=2.5, label=r'$N_{\mathrm{Chinchilla}} \propto C^{0.50}$')
ax.loglog(C_budget, N_kap, color=RED, lw=2.5, ls='--', label=r'$N_{\mathrm{Kaplan}} \propto C^{0.73}$')
ax.loglog(C_budget, D_chin, color=GREEN, lw=2.5, ls=':', label=r'$D_{\mathrm{Chinchilla}} \propto C^{0.50}$')
ax.loglog(C_budget, D_kap, color=PURPLE, lw=2.5, ls='-.', label=r'$D_{\mathrm{Kaplan}} \propto C^{0.27}$')
ax.set_xlabel('Compute budget $C$ (FLOPs)'); ax.set_ylabel('Optimal $N$ or $D$')
ax.set_title('Compute-Optimal Frontier')
ax.legend(fontsize=7, loc='upper left')

# --- Panel 3: Emergence as metric artifact ---
ax = axes[2]
log_N = np.linspace(6, 12, 200)
N_0, k, K = 9.0, 3.0, 4  # inflection, steepness, number of choices
p = 1.0 / (1.0 + np.exp(-k * (log_N - N_0)))  # smooth sigmoid capability

accuracy = np.where(p > 1.0 / K, p, 1.0 / K)  # accuracy: clipped at random chance
log_loss = -np.log(np.clip(p, 1e-12, 1.0))     # log-loss: always smooth

ax2 = ax.twinx()
ax.plot(log_N, accuracy, color=BLUE, lw=2.5, label='Accuracy (nonlinear metric)')
ax.axhline(1.0 / K, color=BLUE, ls=':', alpha=0.4, lw=1)
ax.set_ylabel('Accuracy', color=BLUE)
ax2.plot(log_N, log_loss, color=RED, lw=2.5, ls='--', label='Log-loss (smooth metric)')
ax2.set_ylabel('Log-loss', color=RED)
ax.set_xlabel(r'$\log_{10} N$ (model size)')
ax.set_title('"Emergence" as Metric Artifact\n(Schaeffer et al., 2023)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')

plt.tight_layout()
plt.show()

print("Left: Loss follows power laws across N, D, C — straight lines on log-log plots.")
print("Center: Chinchilla scales N and D equally; Kaplan over-allocates to N.")
print("Right: Same smooth sigmoid capability appears discontinuous under accuracy but smooth under log-loss.")

## 🐍 Implementation from Scratch

We implement three core computations entirely from scratch:
1. **Power-law fitting** — fit $L = A \cdot X^{-\alpha}$ to synthetic scaling data using
   least-squares in log-log space, recovering the exponents.
2. **Chinchilla compute-optimal allocation** — given a compute budget $C$ and the fitted
   loss decomposition $L(N,D) = A N^{-\alpha} + B D^{-\beta} + L_\infty$, solve for
   $(N_{\text{opt}}, D_{\text{opt}})$ using the closed-form from the Lagrange multiplier
   derivation (recall `04.05`).
3. **Schaeffer emergence reproduction** — demonstrate that a smooth sigmoid capability
   composed with a thresholded accuracy metric produces a spurious discontinuity.

In [ ]:
rng = np.random.default_rng(42)

# ========== 1. Power-Law Fitting ==========
print("=" * 60)
print("1. POWER-LAW FITTING ON SYNTHETIC SCALING DATA")
print("=" * 60)

# Generate synthetic scaling data: L(N) = A * N^(-alpha) + noise
true_A, true_alpha = 5.0, 0.076
N_data = np.logspace(6, 11, 30)
L_data = true_A * N_data ** (-true_alpha) * (1 + 0.02 * rng.standard_normal(30))

# Fit in log-log space: log L = log A - alpha * log N
log_N = np.log(N_data)
log_L = np.log(L_data)
# Least-squares: [1, log_N] @ [log_A, -alpha]^T = log_L
X_design = np.column_stack([np.ones(len(log_N)), log_N])
coeffs = np.linalg.lstsq(X_design, log_L, rcond=None)[0]
fitted_A = np.exp(coeffs[0])
fitted_alpha = -coeffs[1]

print(f"True A = {true_A:.3f}, fitted A = {fitted_A:.3f}")
print(f"True alpha = {true_alpha:.4f}, fitted alpha = {fitted_alpha:.4f}")
print(f"Relative error in alpha: {abs(fitted_alpha - true_alpha) / true_alpha * 100:.2f}%")
print()

# ========== 2. Chinchilla Compute-Optimal Allocation ==========
print("=" * 60)
print("2. CHINCHILLA COMPUTE-OPTIMAL ALLOCATION")
print("=" * 60)

def chinchilla_optimal(C, A=406.4, B=410.7, alpha=0.34, beta=0.28, L_inf=1.69):
    """Compute-optimal (N, D) given compute budget C = 6*N*D.
    
    Default constants (A, B, alpha, beta, L_inf) are the fitted values reported
    by Hoffmann et al. (2022), 'Training Compute-Optimal Large Language Models'
    (the Chinchilla paper, cited in Further Reading); they are specific to that
    study's data and tokenizer, not universal.

    Loss model: L(N,D) = A * N^{-alpha} + B * D^{-beta} + L_inf
    Lagrange optimality condition (from Theory section):
        N_opt = G * C^{beta/(alpha+beta)}
    where G = ((alpha * A) / (beta * B))^{1/(alpha+beta)} / 6^{beta/(alpha+beta)}
    """
    a = beta / (alpha + beta)  # exponent for N vs C
    ratio = (alpha * A) / (beta * B)
    G = ratio ** (1 / (alpha + beta)) / 6 ** a
    N_opt = G * C ** a
    D_opt = C / (6 * N_opt)
    L_opt = A * N_opt ** (-alpha) + B * D_opt ** (-beta) + L_inf
    return N_opt, D_opt, L_opt

# Example: 10^21 FLOPs (roughly GPT-3 scale compute)
for C_exp in [18, 21, 24]:
    C = 10.0 ** C_exp
    N_opt, D_opt, L_opt = chinchilla_optimal(C)
    print(f"C = 10^{C_exp} FLOPs:")
    print(f"  N_opt = {N_opt:.2e} params, D_opt = {D_opt:.2e} tokens")
    print(f"  Tokens/param ratio = {D_opt / N_opt:.1f}")
    print(f"  L_opt = {L_opt:.4f} nats")
    print()

# Verify N and D scale equally with C
C_test = np.logspace(18, 25, 100)
N_test = np.array([chinchilla_optimal(c)[0] for c in C_test])
log_fit = np.polyfit(np.log10(C_test), np.log10(N_test), 1)
print(f"Empirical N_opt ~ C^{log_fit[0]:.3f} (Chinchilla predicts ~0.5)")
print()

# ========== 3. Schaeffer Emergence Reproduction ==========
print("=" * 60)
print("3. SCHAEFFER EMERGENCE REPRODUCTION")
print("=" * 60)

def smooth_capability(log_N, k=3.0, N0=9.0):
    """Smooth sigmoid per-token probability."""
    return 1.0 / (1.0 + np.exp(-k * (log_N - N0)))

def accuracy_metric(p, K=4, n_questions=100, n_trials=500, rng=rng):
    """Simulate multiple-choice accuracy: correct iff model's choice is right.
    
    With per-token probability p, probability of getting a K-choice question
    right is approximately p (simplified model).
    """
    # Simulate: for each trial, draw n_questions Bernoulli(p)
    correct = rng.random((n_trials, n_questions)) < p
    return correct.mean(axis=1).mean(axis=0)

log_N_grid = np.linspace(6, 12, 25)
p_vals = smooth_capability(log_N_grid)

acc_vals = np.array([accuracy_metric(p) for p in p_vals])
logloss_vals = -np.log(np.clip(p_vals, 1e-12, 1.0))

print(f"{'log10(N)':>10} {'p(N)':>8} {'Accuracy':>10} {'Log-loss':>10}")
print("-" * 42)
for i in range(0, len(log_N_grid), 4):
    print(f"{log_N_grid[i]:10.1f} {p_vals[i]:8.4f} {acc_vals[i]:10.4f} {logloss_vals[i]:10.4f}")

print()
print("Key observation: log-loss decreases SMOOTHLY throughout,")
print("while accuracy shows a sharp jump around log10(N) ≈ 9.")
print("The 'emergence' is in the METRIC, not the capability.")

## 🤖 Why This Matters for AI

Scaling laws are **the** decision framework for foundation-model training. Every major AI lab
uses them to plan training runs costing $10^6$–$10^8$ dollars:

- **Budget allocation**: Given a fixed GPU-hour budget, the Chinchilla formula tells you
  exactly how many parameters to use and how many tokens to train on. Getting this wrong
  wastes millions of dollars — GPT-3 was $\approx 3\times$ over-parameterized by Chinchilla
  standards.
- **Performance prediction**: Before training a 100B-parameter model ($\sim$\$10M+), you train
  a sequence of smaller models (1M, 10M, 100M, 1B params), fit the power law, and
  *extrapolate* to predict the large model's loss. If the predicted loss isn't good enough,
  you don't train the big model at all.
- **Diminishing returns**: The small exponents ($\alpha \approx 0.05$–$0.10$) mean each
  $10\times$ more compute yields only $\sim 10$–$20\%$ loss reduction. This is the
  fundamental economic constraint on scaling.
- **Emergence claims**: Understanding the metric-artifact argument from Schaeffer et al. is
  critical for evaluating whether a model has genuinely new capabilities at scale, or whether
  the evaluation metric is creating an illusion of discontinuity.

Below, we simulate a realistic scenario: a team has a budget of 1000 H100 GPUs for 30 days
and needs to decide how to allocate compute between model size and training data.

In [ ]:
# Real-world scenario: compute-optimal planning for an LLM training run

# Hardware specs
n_gpus = 1000
days = 30
hours_per_day = 24
gpu_utilization = 0.40  # typical MFU (model FLOPs utilization)
h100_peak_flops = 989e12  # H100 SXM peak BF16 FLOPS

total_seconds = days * hours_per_day * 3600
effective_flops_per_gpu = h100_peak_flops * gpu_utilization
C_total = n_gpus * effective_flops_per_gpu * total_seconds

print("=" * 60)
print("COMPUTE-OPTIMAL TRAINING PLAN")
print("=" * 60)
print(f"Hardware: {n_gpus} H100 GPUs x {days} days")
print(f"GPU utilization (MFU): {gpu_utilization:.0%}")
print(f"Total compute: {C_total:.2e} FLOPs = 10^{np.log10(C_total):.1f} FLOPs")
print()

# Chinchilla-optimal allocation
N_opt, D_opt, L_opt = chinchilla_optimal(C_total)
tokens_per_param = D_opt / N_opt

print("--- Chinchilla-optimal allocation ---")
print(f"Model size: {N_opt:.2e} params ({N_opt / 1e9:.1f}B)")
print(f"Training tokens: {D_opt:.2e} ({D_opt / 1e12:.2f}T)")
print(f"Tokens/param ratio: {tokens_per_param:.1f}")
print(f"Predicted loss: {L_opt:.4f} nats")
print()

# Compare with Kaplan-style over-parameterized allocation
# Kaplan: N ~ C^0.73, leaving less for D
N_kaplan = N_opt * 3.0  # roughly 3x over-parameterized
D_kaplan = C_total / (6 * N_kaplan)
A, B, alpha, beta, L_inf = 406.4, 410.7, 0.34, 0.28, 1.69
L_kaplan = A * N_kaplan ** (-alpha) + B * D_kaplan ** (-beta) + L_inf

print("--- Kaplan-style (3x over-parameterized) ---")
print(f"Model size: {N_kaplan:.2e} params ({N_kaplan / 1e9:.1f}B)")
print(f"Training tokens: {D_kaplan:.2e} ({D_kaplan / 1e12:.2f}T)")
print(f"Tokens/param ratio: {D_kaplan / N_kaplan:.1f}")
print(f"Predicted loss: {L_kaplan:.4f} nats")
print(f"Loss penalty vs optimal: +{L_kaplan - L_opt:.4f} nats ({(L_kaplan - L_opt) / (L_opt - L_inf) * 100:.1f}% worse reducible loss)")
print()

# Scaling prediction: fit small models, extrapolate
print("--- Scaling prediction from small runs ---")
small_C = C_total * np.array([1e-6, 1e-5, 1e-4, 1e-3, 1e-2])
small_N = np.array([chinchilla_optimal(c)[0] for c in small_C])
small_L = np.array([chinchilla_optimal(c)[2] for c in small_C])

# Fit power law to small runs: L - L_inf = A' * C^(-alpha_C)
log_C_small = np.log(small_C)
log_L_red_small = np.log(small_L - L_inf)  # reducible loss
fit_coeffs = np.polyfit(log_C_small, log_L_red_small, 1)
predicted_log_L_red = np.polyval(fit_coeffs, np.log(C_total))
predicted_L = np.exp(predicted_log_L_red) + L_inf

print(f"Fitted scaling exponent from small runs: {-fit_coeffs[0]:.4f}")
print(f"Extrapolated loss at full scale: {predicted_L:.4f} nats")
print(f"Actual (formula) loss at full scale: {L_opt:.4f} nats")
print(f"Prediction error: {abs(predicted_L - L_opt):.6f} nats")
print()
print("This demonstrates the practical workflow: train small, fit power law, decide")
print("whether the big run is worth the cost BEFORE spending $10M+.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using the Chinchilla formula with the parameters from the Implementation section
   ($A=406.4$, $B=410.7$, $\alpha=0.34$, $\beta=0.28$, $L_\infty=1.69$), compute
   $N_{\text{opt}}$ and $D_{\text{opt}}$ for a compute budget of $C = 10^{21}$ FLOPs.
   Verify that $6 N_{\text{opt}} D_{\text{opt}} \approx C$.

<details>
<summary>💡 Solution</summary>

From the Lagrange derivation, $a = \beta / (\alpha + \beta) = 0.28 / 0.62 \approx 0.4516$.
The proportionality constant $G = ((\alpha A)/(\beta B))^{1/(\alpha+\beta)} / 6^a$.
Computing: $\alpha A / (\beta B) = (0.34 \times 406.4)/(0.28 \times 410.7) \approx 1.2012$.
Then $G = 1.2012^{1/0.62} / 6^{0.4516} \approx 1.345 / 2.553 \approx 0.527$.
So $N_{\text{opt}} = 0.527 \times (10^{21})^{0.4516} \approx 0.527 \times 10^{9.48} \approx 1.59 \times 10^9$
(about 1.6B parameters).
$D_{\text{opt}} = 10^{21} / (6 \times 1.59 \times 10^9) \approx 1.05 \times 10^{11}$ (about 105B tokens).
Check: $6 \times 1.59 \times 10^9 \times 1.05 \times 10^{11} \approx 10^{21}$ ✓.
Running `chinchilla_optimal(1e21)` confirms these values.

</details>

### 💭 UNDERSTAND
2. Kaplan et al. recommended scaling model size much faster than data ($N \propto C^{0.73}$,
   $D \propto C^{0.27}$), while Chinchilla found $N \propto C^{0.5}$, $D \propto C^{0.5}$.
   Why do the two studies disagree? The answer isn't "one is wrong" — it relates to an
   experimental design choice. What was it, and how does it bias the fitted exponents?

<details>
<summary>💡 Solution</summary>

Kaplan et al. used a fixed, *small* number of training tokens for each model size, stopping
well before convergence at larger scales. This means the larger models were systematically
undertrained — they hadn't yet reached their optimal loss for the given $N$. The data thus
reflected the loss achievable with model size $N$ *at a fixed, insufficient $D$*, not the
true $L(N)$ at optimal $D$. This makes the $N$-scaling curve steeper than it should be
(more benefit from bigger $N$ when $D$ is inadequate, because the loss bottleneck is $N$
not $D$), and simultaneously makes the $D$-scaling curve shallower (less apparent benefit
from more data because the model is already too small to use it). The result: overstating
$\alpha_N$ and understating $\alpha_D$, leading to the recommendation to scale $N$ faster.
Hoffmann et al. (Chinchilla) corrected this by fitting across a grid where each $(N, D)$
combination included *converged* training runs, revealing the true balanced scaling.

</details>

### ✏️ DERIVE
3. Starting from the loss decomposition $L(N, D) = A N^{-\alpha} + B D^{-\beta} + L_\infty$
   and the compute constraint $C = 6ND$, use the method of Lagrange multipliers (recall
   `04.05`) to derive the compute-optimal exponent $a = \beta / (\alpha + \beta)$ in
   $N_{\text{opt}} \propto C^a$. Show all steps.

<details>
<summary>💡 Solution</summary>

Form the Lagrangian: $\mathcal{L} = A N^{-\alpha} + B D^{-\beta} + L_\infty + \lambda(6ND - C)$.

Stationarity conditions:
$$\frac{\partial \mathcal{L}}{\partial N} = -A\alpha N^{-\alpha-1} + 6\lambda D = 0 \implies \lambda = \frac{A\alpha N^{-\alpha-1}}{6D}$$
$$\frac{\partial \mathcal{L}}{\partial D} = -B\beta D^{-\beta-1} + 6\lambda N = 0 \implies \lambda = \frac{B\beta D^{-\beta-1}}{6N}$$

Equating: $A\alpha N^{-\alpha-1} / D = B\beta D^{-\beta-1} / N$, so $A\alpha N^{-\alpha} = B\beta D^{-\beta}$.

Substitute $D = C/(6N)$:
$$A\alpha N^{-\alpha} = B\beta \left(\frac{C}{6N}\right)^{-\beta} = B\beta \cdot \frac{(6N)^\beta}{C^\beta}$$

$$A\alpha N^{-\alpha} = B\beta \cdot 6^\beta N^\beta C^{-\beta}$$

$$N^{-\alpha - \beta} = \frac{B\beta \cdot 6^\beta}{A\alpha} C^{-\beta}$$

$$N^{\alpha + \beta} = \frac{A\alpha}{B\beta \cdot 6^\beta} C^\beta$$

$$N = \left(\frac{A\alpha}{B\beta \cdot 6^\beta}\right)^{1/(\alpha+\beta)} C^{\beta/(\alpha+\beta)}$$

Therefore $a = \beta/(\alpha + \beta)$. With $\alpha \approx \beta$, this gives $a \approx 0.5$. $\square$

</details>

### 🐍 IMPLEMENT
4. Fit a power law $L = A \cdot N^{-\alpha} + L_\infty$ to the following synthetic data
   (generated from a known ground truth with noise). Use `scipy.optimize.curve_fit` on
   the raw (non-log) data and compare your fitted $\alpha$ to the log-log least-squares
   method used in the Implementation cell. Which gives a better fit and why?

```python
N_ex = np.logspace(6, 11, 20)
L_ex = 5.0 * N_ex ** (-0.076) + 1.69 + 0.01 * np.random.default_rng(123).standard_normal(20)
```

<details>
<summary>💡 Solution</summary>

The log-log method implicitly fits $\log(L - L_\infty) = \log A - \alpha \log N$, which
requires knowing (or correctly estimating) $L_\infty$ — subtracting the wrong irreducible
loss biases $\alpha$. `curve_fit` on the raw model $f(N) = A N^{-\alpha} + L_\infty$ with
three free parameters avoids this issue and typically recovers $\alpha$ more accurately,
though it requires good initial guesses to avoid local minima (start with `p0=[5, 0.08, 1.5]`).
The log-log method is faster and more robust when $L_\infty$ is known; `curve_fit` is better
when all three parameters must be estimated jointly.

</details>

### 🤖 APPLY
5. You have 1000 H100 GPUs for 30 days (as in the AI section). Your boss asks: "Can we
   match GPT-4's estimated performance ($L \approx 1.85$ nats) with this budget?" Using
   the Chinchilla formula, determine whether the compute-optimal loss at this budget reaches
   the target. If not, estimate how much compute (and thus how many GPU-days) would be needed.

<details>
<summary>💡 Solution</summary>

From the AI section, 1000 H100s × 30 days gives $C \approx 10^{24.4}$ FLOPs. Running
`chinchilla_optimal(C)` gives $L_{\text{opt}} \approx 1.79$ nats, which is *below* the
target of 1.85 — so yes, this budget is theoretically sufficient (with Chinchilla-optimal
allocation). However, the loss model assumes perfect scaling with no overhead; real-world
factors (communication overhead, data quality, hyperparameter tuning compute) would eat into
this margin. A conservative estimate would add $\sim 20\%$ compute overhead, raising the
effective loss slightly. The key message: scaling laws provide a *lower bound* on achievable
loss, not a guarantee — but they're accurate enough to make go/no-go decisions.

</details>

### 🚀 CHALLENGE
6. **Demonstrate emergence as a metric artifact.** Generate a smooth sigmoid capability
   curve $p(N)$ and measure it with three different metrics: (a) exact-match accuracy on
   8-token sequences ($p^8$, sharply thresholded), (b) per-token accuracy ($p$, smooth),
   (c) Brier score ($\mathbb{E}[(1-p)^2]$, smooth). Show that (a) exhibits a sharp "phase
   transition" while (b) and (c) are smooth — and quantify the location and width of the
   apparent transition in (a) vs the inflection point of the underlying sigmoid.

<details>
<summary>🔍 What a strong answer covers</summary>

The 8-token exact match $p^8$ is a *much* sharper sigmoid than $p$ itself: if $p = 0.9$,
exact match is $0.9^8 \approx 0.43$; if $p = 0.95$, exact match is $0.95^8 \approx 0.66$;
if $p = 0.99$, exact match is $0.99^8 \approx 0.92$. The transition from near-zero to
near-one in exact match accuracy is compressed into a much narrower range of $\log N$ than
the underlying sigmoid, creating a visually dramatic "emergence." Per-token accuracy and
Brier score, being smooth functions of $p$, show no such discontinuity. The apparent
transition in (a) is centered near the inflection point of $p(N)$ (at $\log N = N_0$) but
its width is narrowed by a factor of $\sim 1/8$ — the more tokens in the sequence, the
sharper the apparent emergence, even though the underlying capability improves at the same
gradual rate throughout. This is exactly Schaeffer et al.'s argument applied to a concrete,
controllable setting.

</details>

---

## 📚 Further Reading
- Kaplan et al., ["Scaling Laws for Neural Language Models"](https://arxiv.org/abs/2001.08361) (2020)
- Hoffmann et al., ["Training Compute-Optimal Large Language Models"](https://arxiv.org/abs/2203.15556) (2022) — the Chinchilla paper
- Schaeffer et al., ["Are Emergent Abilities of Large Language Models a Mirage?"](https://arxiv.org/abs/2304.15004) (2023)
- Henighan et al., ["Scaling Laws for Autoregressive Generative Modeling"](https://arxiv.org/abs/2010.14701) (2020) — scaling beyond language
- Bahri et al., ["Explaining Neural Scaling Laws"](https://arxiv.org/abs/2102.06701) (2021) — statistical mechanics connection
- Zhai et al., ["Scaling Vision Transformers"](https://arxiv.org/abs/2106.04560) (2022) — scaling laws for vision

---

*Next notebook: End of Module 13*